# 05 - Modelos Avanzados para Predicción de Precios de Vivienda

En este notebook implementaremos modelos avanzados de machine learning para mejorar la precisión en la predicción de precios de vivienda.

## Modelos a implementar:
- **Random Forest**: Ensamble de árboles de decisión
- **XGBoost**: Gradient boosting optimizado
- **Support Vector Regression (SVR)**: Máquinas de vectores de soporte
- **k-Nearest Neighbors (KNN)**: Vecinos más cercanos

## Objetivos:
1. Implementar y optimizar modelos avanzados
2. Comparar su rendimiento con modelos lineales
3. Analizar la importancia de características
4. Seleccionar el mejor modelo para el problema

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Cargar datos procesados

In [ ]:
# Cargar datos de entrenamiento y prueba
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test = pd.read_csv('../data/processed/X_test_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').iloc[:, 0]
y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0]

print(f"Datos de entrenamiento: {X_train.shape}")
print(f"Datos de prueba: {X_test.shape}")
print(f"Características disponibles: {list(X_train.columns)}")

## 2. Función de evaluación de modelos

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Evalúa un modelo de regresión y retorna métricas
    """
    # Entrenar modelo
    model.fit(X_train, y_train)
    
    # Predicciones
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Métricas
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    cv_r2_mean = cv_scores.mean()
    cv_r2_std = cv_scores.std()
    
    # Crear dataframe de resultados
    results = {
        'Model': model_name,
        'Train_RMSE': train_rmse,
        'Test_RMSE': test_rmse,
        'Train_MAE': train_mae,
        'Test_MAE': test_mae,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'CV_R2_Mean': cv_r2_mean,
        'CV_R2_Std': cv_r2_std,
        'Overfitting': train_r2 - test_r2
    }
    
    return results, y_train_pred, y_test_pred, model

## 3. Random Forest Regressor

Random Forest es un método de ensamble que construye múltiples árboles de decisión y promedia sus predicciones. Es robusto contra overfitting y puede capturar relaciones no lineales.

In [ ]:
# Definir hiperparámetros para Random Forest
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Crear modelo base
rf_model = RandomForestRegressor(random_state=42)

# Grid Search
print("Optimizando Random Forest...")
rf_grid = GridSearchCV(rf_model, rf_params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {rf_grid.best_params_}")
print(f"Mejor score CV: {rf_grid.best_score_:.4f}")

# Evaluar mejor modelo
rf_results, rf_train_pred, rf_test_pred, rf_best_model = evaluate_model(
    rf_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Random Forest'
)

print(f"Random Forest - Test R²: {rf_results['Test_R2']:.4f}")
print(f"Random Forest - Test RMSE: {rf_results['Test_RMSE']:.2f}")

In [ ]:
# Analizar importancia de características
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_best_model.feature_importances_
}).sort_values('importance', ascending=False)

# Visualizar importancia
plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
plt.xlabel('Importancia')
plt.title('Top 15 Características más Importantes - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 características más importantes:")
print(feature_importance.head(10))

## 4. XGBoost (Gradient Boosting)

XGBoost es una implementación optimizada de gradient boosting que construye árboles de forma secuencial, corrigiendo errores de iteraciones anteriores.

In [ ]:
# Intentar importar XGBoost
try:
    import xgboost as xgb
    xgb_available = True
except ImportError:
    print("XGBoost no está instalado. Usando GradientBoostingRegressor de sklearn.")
    xgb_available = False
    from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
if xgb_available:
    # XGBoost
    xgb_model = xgb.XGBRegressor(random_state=42)
    xgb_params = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0]
    }
else:
    # Usar GradientBoostingRegressor si XGBoost no está disponible
    xgb_model = GradientBoostingRegressor(random_state=42)
    xgb_params = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0]
    }

# Grid Search
print("Optimizando XGBoost/Gradient Boosting...")
xgb_grid = GridSearchCV(xgb_model, xgb_params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
xgb_grid.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {xgb_grid.best_params_}")
print(f"Mejor score CV: {xgb_grid.best_score_:.4f}")

# Evaluar mejor modelo
xgb_results, xgb_train_pred, xgb_test_pred, xgb_best_model = evaluate_model(
    xgb_grid.best_estimator_, X_train, X_test, y_train, y_test, 'XGBoost' if xgb_available else 'Gradient Boosting'
)

print(f"XGBoost - Test R²: {xgb_results['Test_R2']:.4f}")
print(f"XGBoost - Test RMSE: {xgb_results['Test_RMSE']:.2f}")

## 5. Support Vector Regression (SVR)

SVR utiliza el principio de máquinas de vectores de soporte para regresión, encontrando un hiperplano que minimiza el error dentro de un margen de tolerancia.

In [ ]:
# SVR requiere datos estandarizados
# Crear modelo SVR
svr_model = SVR()

# Hiperparámetros para SVR
svr_params = {
    'kernel': ['rbf', 'linear'],
    'C': [0.1, 1, 10],
    'epsilon': [0.1, 0.5, 1.0]
}

# Grid Search (usar menos combinaciones por tiempo de cómputo)
print("Optimizando SVR...")
svr_grid = GridSearchCV(svr_model, svr_params, cv=3, scoring='r2', n_jobs=-1, verbose=1)
svr_grid.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {svr_grid.best_params_}")
print(f"Mejor score CV: {svr_grid.best_score_:.4f}")

# Evaluar mejor modelo
svr_results, svr_train_pred, svr_test_pred, svr_best_model = evaluate_model(
    svr_grid.best_estimator_, X_train, X_test, y_train, y_test, 'SVR'
)

print(f"SVR - Test R²: {svr_results['Test_R2']:.4f}")
print(f"SVR - Test RMSE: {svr_results['Test_RMSE']:.2f}")

## 6. k-Nearest Neighbors (KNN) Regressor

KNN predice el valor de una instancia basándose en los valores de sus k vecinos más cercanos en el espacio de características.

In [ ]:
# Crear modelo KNN
knn_model = KNeighborsRegressor()

# Hiperparámetros para KNN
knn_params = {
    'n_neighbors': [3, 5, 7, 10, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Grid Search
print("Optimizando KNN...")
knn_grid = GridSearchCV(knn_model, knn_params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
knn_grid.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {knn_grid.best_params_}")
print(f"Mejor score CV: {knn_grid.best_score_:.4f}")

# Evaluar mejor modelo
knn_results, knn_train_pred, knn_test_pred, knn_best_model = evaluate_model(
    knn_grid.best_estimator_, X_train, X_test, y_train, y_test, 'KNN'
)

print(f"KNN - Test R²: {knn_results['Test_R2']:.4f}")
print(f"KNN - Test RMSE: {knn_results['Test_RMSE']:.2f}")

## 7. Comparación de Modelos Avanzados

In [ ]:
# Consolidar resultados
results_df = pd.DataFrame([rf_results, xgb_results, svr_results, knn_results])

# Mostrar tabla de resultados
print("Comparación de Modelos Avanzados:")
print("="*80)
display(results_df.round(4))

# Identificar mejor modelo
best_model_idx = results_df['Test_R2'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
print(f"\nMejor modelo: {best_model_name} (R² = {results_df.loc[best_model_idx, 'Test_R2']:.4f})")

In [ ]:
# Visualizar comparación de métricas
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# R² Score
axes[0,0].bar(results_df['Model'], results_df['Test_R2'], color='skyblue')
axes[0,0].set_title('R² Score por Modelo')
axes[0,0].set_ylabel('R² Score')
axes[0,0].tick_params(axis='x', rotation=45)

# RMSE
axes[0,1].bar(results_df['Model'], results_df['Test_RMSE'], color='lightcoral')
axes[0,1].set_title('RMSE por Modelo')
axes[0,1].set_ylabel('RMSE')
axes[0,1].tick_params(axis='x', rotation=45)

# MAE
axes[1,0].bar(results_df['Model'], results_df['Test_MAE'], color='lightgreen')
axes[1,0].set_title('MAE por Modelo')
axes[1,0].set_ylabel('MAE')
axes[1,0].tick_params(axis='x', rotation=45)

# Overfitting
axes[1,1].bar(results_df['Model'], results_df['Overfitting'], color='gold')
axes[1,1].set_title('Overfitting (Train R² - Test R²)')
axes[1,1].set_ylabel('Diferencia R²')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Análisis de Residuos del Mejor Modelo

In [ ]:
# Seleccionar el mejor modelo para análisis detallado
if best_model_name == 'Random Forest':
    best_model = rf_best_model
    best_test_pred = rf_test_pred
elif best_model_name == 'XGBoost':
    best_model = xgb_best_model
    best_test_pred = xgb_test_pred
elif best_model_name == 'SVR':
    best_model = svr_best_model
    best_test_pred = svr_test_pred
else:
    best_model = knn_best_model
    best_test_pred = knn_test_pred

# Calcular residuos
residuos = y_test - best_test_pred

# Visualizar residuos
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Residuos vs Predichos
axes[0,0].scatter(best_test_pred, residuos, alpha=0.6)
axes[0,0].axhline(y=0, color='r', linestyle='--')
axes[0,0].set_xlabel('Valores Predichos')
axes[0,0].set_ylabel('Residuos')
axes[0,0].set_title(f'Residuos vs Predichos - {best_model_name}')

# Histograma de residuos
axes[0,1].hist(residuos, bins=30, edgecolor='black', alpha=0.7)
axes[0,1].set_xlabel('Residuos')
axes[0,1].set_ylabel('Frecuencia')
axes[0,1].set_title(f'Histograma de Residuos - {best_model_name}')

# Q-Q plot
from scipy import stats
stats.probplot(residuos, dist="norm", plot=axes[1,0])
axes[1,0].set_title(f'Q-Q Plot - {best_model_name}')

# Valores reales vs predichos
axes[1,1].scatter(y_test, best_test_pred, alpha=0.6)
axes[1,1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1,1].set_xlabel('Valores Reales')
axes[1,1].set_ylabel('Valores Predichos')
axes[1,1].set_title(f'Reales vs Predichos - {best_model_name}')

plt.tight_layout()
plt.show()

# Estadísticas de residuos
print(f"Estadísticas de residuos para {best_model_name}:")
print(f"Media: {residuos.mean():.4f}")
print(f"Desviación estándar: {residuos.std():.4f}")
print(f"MSE de residuos: {np.mean(residuos**2):.4f}")

## 9. Guardar Mejores Modelos

In [ ]:
import joblib
import os

# Crear directorio para modelos si no existe
os.makedirs('../models', exist_ok=True)

# Guardar modelos avanzados
joblib.dump(rf_best_model, '../models/random_forest_model.pkl')
joblib.dump(xgb_best_model, '../models/xgboost_model.pkl')
joblib.dump(svr_best_model, '../models/svr_model.pkl')
joblib.dump(knn_best_model, '../models/knn_model.pkl')

# Guardar resultados
results_df.to_csv('../reports/advanced_models_results.csv', index=False)

print("Modelos guardados exitosamente!")
print(f"Mejor modelo: {best_model_name}")
print(f"R² Score: {results_df.loc[best_model_idx, 'Test_R2']:.4f}")
print(f"RMSE: {results_df.loc[best_model_idx, 'Test_RMSE']:.2f}")

## 10. Conclusiones y Recomendaciones

### Resultados Principales:

1. **Mejor Modelo**: {} con R² = {:.4f}
2. **Segundo Mejor**: {} con R² = {:.4f}
3. **RMSE más bajo**: {} con RMSE = {:.2f}

### Observaciones:
- Los modelos de ensamble (Random Forest, XGBoost) mostraron mejor rendimiento
- SVR y KNN tuvieron un desempeño moderado
- La importancia de características reveló que {} son las variables más influyentes
- El análisis de residuos muestra {}

### Recomendaciones para el siguiente paso:
1. En el notebook 06, compararemos todos los modelos (lineales y avanzados)
2. Implementaremos técnicas de ensemble avanzadas
3. Prepararemos el modelo final para despliegue
4. Crearemos un pipeline completo de predicción